# 2.3 The ReAct Pattern

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

## Your API key

Some cells in this notebook call a model and need your own OpenAI API key. Paste it when asked. It is stored only in this notebook session, never saved to the file.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Learning objectives

- Explain the ReAct loop (thought, action, observation) and why it works as an agent's control flow.
- Write a minimal ReAct loop by hand with a prompt, `bind_tools`, and manual tool execution.
- Build the same agent with `create_agent` from LangChain 1.x and name what the harness gives you for free.
- Decide when to hand-roll a loop and when to use `create_agent`.
- Explain why `AgentExecutor` is legacy and which import path the current API uses.

## The loop in plain words

ReAct stands for Reasoning + Acting, and the mechanism is a loop with three steps that repeats until the job is done:

1. **Thought.** The model looks at the conversation so far and reasons, in words, about what to do next. ("The user wants a total with tax. I have the subtotal from the earlier message. I need the tax rate, so I should call the calculator with 120 * 1.19.")
2. **Action.** The model emits a tool call: which tool, with which arguments.
3. **Observation.** Your code runs the tool and appends the result to the conversation as a `ToolMessage`.

Then the loop repeats. The next thought reads the new observation, and eventually the model produces a final answer instead of another tool call. That is the whole pattern. Everything else in this module is plumbing around these three steps.

Why does this work so well? Because the reasoning step gives the model a scratchpad. Without it, the model must jump straight from the question to the final answer, which is where hallucinations live. With it, each action is grounded in fresh observations, and each observation is checked by the next thought. The loop also bounds the problem: the model never has to plan ten steps ahead, only one.

## Seeing the loop: hand-rolled ReAct

You should write this loop once by hand, because it demystifies everything `create_agent` does later. The ingredients: a system prompt that teaches the model the loop discipline, a model with tools bound via `bind_tools`, and a small driver that executes tool calls and feeds results back.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

@tool
def calculate(expression: str) -> str:
    """Evaluate a simple arithmetic expression.

    Use this for any math instead of doing mental arithmetic.
    Args:
        expression: A Python arithmetic expression, e.g. '38 + 47'.
    """
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception:
        return f"Error: could not evaluate '{expression}'. Send a plain arithmetic expression."

@tool
def get_current_time() -> str:
    """Return the current UTC date and time as an ISO string.

    Use this whenever the user asks about 'now', 'today', or the current date.
    """
    from datetime import datetime, timezone
    return datetime.now(timezone.utc).isoformat()

tools = [calculate, get_current_time]
by_name = {t.name: t for t in tools}

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

messages = [
    SystemMessage(
        "You are a helpful assistant. When you need a calculation, use the "
        "calculate tool instead of doing arithmetic yourself. When you need the "
        "current date or time, use get_current_time. Answer only when you have "
        "everything you need."
    ),
    HumanMessage("What is 38 + 47? And what is today's date?"),
]

for step in range(6):                       # loop cap: never loop forever
    ai_msg = llm_with_tools.invoke(messages)
    messages.append(ai_msg)

    if ai_msg.content:
        print(f"[thought] {ai_msg.content}")

    if not ai_msg.tool_calls:               # no tool calls: the model is done
        print(f"[final] {ai_msg.content}")
        break

    for call in ai_msg.tool_calls:
        print(f"[action] {call['name']}({call['args']})")
        result = by_name[call["name"]].invoke(call)   # dict in, ToolMessage out
        print(f"[observation] {result.content}")
        messages.append(result)

Run it and the printed trace shows the pattern breathing: a thought, one or two tool calls, observations, then a final answer that uses both results. Three details worth noticing in the code:

- `llm.bind_tools(tools)` attaches the tools' JSON schemas to the model call. It returns a runnable; it does not change the original `llm`.
- `by_name[call["name"]].invoke(call)` takes the whole tool-call dict (name, args, id) and returns a `ToolMessage` carrying the original `id`. Passing the dict, not just the args, is what keeps the tool result linked to the right call. This is the current supported pattern in langchain-core.
- The `range(6)` cap is load-bearing, not decorative. Without a cap, a confused model loops until your budget does. Every hand-rolled loop needs one.

This loop is correct, small, and honest. It is also missing everything a production agent needs: retries when a tool crashes, streaming for a responsive UI, a way to pause before a dangerous tool runs, persistence across restarts, and a way to enforce a budget. That gap is what `create_agent` fills.

## The production fast path: create_agent

In LangChain 1.x, `create_agent` is the supported way to build a standard tool-calling agent. It lives in the `langchain.agents` package and returns a compiled LangGraph graph, which means the agent runs on the LangGraph runtime: you get persistence, streaming, and human-in-the-loop support from the same object.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
# tools: calculate, get_current_time from lesson 2.2

agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini"),
    tools=[calculate, get_current_time],
    system_prompt=(
        "You are a helpful assistant. Use the calculate tool for any math "
        "instead of doing arithmetic yourself. Use get_current_time when the "
        "user asks about the date or time."
    ),
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is 38 + 47? And what is today's date?"}]}
)
print(result["messages"][-1].content)

The input and output are message lists, exactly like the hand-rolled version: the graph appends AIMessages, tool calls, and ToolMessages as it runs, and `result["messages"]` holds the full trace. If you pass `model="openai:gpt-4o-mini"` as a string instead of an instance, `create_agent` builds the chat model for you; passing an instance gives you control over temperature, retries, and other knobs.

Two parameter notes that bite people migrating from older code:

- The parameter is `system_prompt=`, renamed from `prompt=` in LangChain 1.0. Old tutorials passing `prompt=` will fail with a type error.
- `create_react_agent` from `langgraph.prebuilt` still exists, but it is the legacy path now. New code uses `langchain.agents.create_agent`.

## What create_agent gives you for free

Compare the eight-line hand-rolled loop with what the one `create_agent` call includes:

- **The loop itself**, with the thought/action/observation cycle handled by the graph. You never write the `while` or the message plumbing.
- **Error handling on tool failures.** When a tool raises, the agent converts the failure into an observation the model can read, so the model retries with corrected arguments instead of the run crashing. (Combine this with the friendly-error-string discipline from lesson 2.2 and the loop is genuinely self-repairing.)
- **Streaming.** `agent.stream({"messages": [...]}, stream_mode="messages")` yields token-level updates, and `astream_events` gives you the structured event log you met in Module 1. Same Runnable conventions, no extra wiring.
- **A loop budget.** Infinite loops are a cost bug, not a logic bug. Two current ways to cap them: the LangGraph `recursion_limit` config (`agent.invoke(input, config={"recursion_limit": 25})`), and the purpose-built `ModelCallLimitMiddleware`, which caps model calls per run or per thread:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini"),
    tools=[calculate, get_current_time],
    system_prompt="You are a helpful assistant.",
    middleware=[ModelCallLimitMiddleware(run_limit=10)],
)

- **Middleware hooks.** The `middleware=[...]` list is the extension system: built-ins cover human-in-the-loop approval of tool calls, tool retries, context pruning, and call limits; custom middleware can wrap model calls or tool calls to add logging, guardrails, or routing. You saw in lesson 2.2 that error handling lives in the tool; middleware is where cross-cutting concerns live instead.
- **The LangGraph runtime.** Because the agent is a compiled graph, you can attach a checkpointer for memory across turns, interrupt before a tool executes for human approval, and resume after a restart. Module 4 builds on this directly; for now, know that the capability is there without rewriting the agent.

## When to hand-roll, when to use create_agent

Use `create_agent` when the shape of the problem is the standard ReAct loop: one model, a set of tools, maybe some middleware. That covers most real agents: support bots, research assistants, data lookup agents. You get the loop, the error handling, the streaming, and the runtime for almost no code.

Hand-roll the loop when the control flow is not standard ReAct. Examples: a classifier must run before any tool is considered; two tools must run in parallel and their results merged before the next thought; a validation step must approve the model's plan before any action executes; or you need a custom stop condition the model should not control. The hand-rolled version from this lesson is your template: you own every line, which is exactly the point.

A practical rule from production: start with `create_agent`, and only hand-roll when you can point at a specific behaviour the harness cannot express. Most "custom agent frameworks" in the wild are `create_agent` with extra steps and extra bugs.

One more migration note, since you will meet it in old tutorials: `AgentExecutor` and the `initialize_agent` helpers are deprecated and live in the `langchain-classic` maintenance package. They are not taught in this course and should not appear in new code. If a tutorial's agent imports fail on your pinned versions, check whether it is an `AgentExecutor` tutorial; if so, translate the example to `create_agent` rather than installing `langchain-classic`.

## What can go wrong

- **Infinite loops.** The model calls a tool, misreads the observation, and calls the same tool with the same arguments forever. Each iteration costs a model call. Always set a budget: `ModelCallLimitMiddleware(run_limit=...)` or `recursion_limit`, plus a max-steps cap on any hand-rolled loop.
- **The model answers without calling any tool.** Usually a weak system prompt or a weak tool description, not a model failure. Strengthen the "when to use it" sentence in the tool description and add an explicit instruction in the system prompt. Check the trace first: `result["messages"]` shows you whether the model even considered the tool.
- **Hallucinated tool arguments.** Covered in lesson 2.2, but it shows up here as the loop spinning: bad args, error observation, bad args again. In-tool validation with format guidance in the error message breaks the cycle.
- **Tools with side effects inside a retry loop.** A tool that sends an email or writes to a database can execute twice when the model retries. Idempotency keys, confirmation arguments, and human-in-the-loop middleware (Module 6) are the defences; hope is not one.
- **Cost per iteration surprises.** A loop with 10 iterations is 10 model calls, each carrying the full conversation history including every tool observation. Verbose tool outputs compound fast. Keep tool outputs tight and set call limits before the first production run.
- **Debugging the graph as a black box.** When `create_agent` misbehaves, do not stare at the final answer. Read `result["messages"]` for the full trace, or stream with `stream_mode="messages"` to watch the loop live. The hand-rolled version taught you what a healthy trace looks like, so an unhealthy one stands out.
- **Stale tutorial imports.** `from langchain.agents import initialize_agent`, `AgentExecutor`, `langgraph.prebuilt.create_react_agent` with `state_modifier`: all legacy. Current path is `from langchain.agents import create_agent` with `system_prompt=`. Pin versions and treat pre-2026 agent tutorials as historical documents.

## Check your understanding

**1. Describe the ReAct loop's three steps and say which one is code and which ones are the model.**
Thought (the model reasons in words about the next step), action (the model emits a tool call), observation (your code runs the tool and appends the result as a `ToolMessage`). Thought and action are model outputs; observation is your code executing. The loop repeats until the model emits a final answer with no tool calls.

**2. In the hand-rolled loop, why do you pass the whole tool-call dict to `tool.invoke(call)` instead of just the args?**
The dict carries the tool call's `id` alongside name and args, and invoking with it returns a `ToolMessage` linked to that id. That linkage is how the model (and the harness) knows which observation answers which call; passing bare args breaks the pairing.

**3. `create_agent` runs on the LangGraph runtime. Name three things that buys you beyond the raw ReAct loop.**
Persistence across turns via checkpointers, streaming and structured event logs through the standard Runnable interface, and human-in-the-loop interrupts before tool execution, plus middleware hooks for guardrails, retries, and call limits.

**4. A production agent must never make more than 10 model calls in one run. Show how, and say why a bare `while True` loop is a cost bug.**
Pass `middleware=[ModelCallLimitMiddleware(run_limit=10)]` to `create_agent`, or set `config={"recursion_limit": 25}` on invoke. A bare loop has no budget: a confused model repeating the same tool call costs a full model call per iteration, each carrying the whole growing history.

**5. Your codebase has `AgentExecutor` code from a 2025 tutorial. What do you do, and why not just install `langchain-classic`?**
Rewrite it with `langchain.agents.create_agent` (and `system_prompt=` instead of `prompt=`). `AgentExecutor` is deprecated in the `langchain-classic` maintenance package: it still runs but receives no new features, and every new concept in this course (middleware, the LangGraph runtime, current tool patterns) assumes `create_agent`. Installing the legacy package keeps the code working while freezing it in the past.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.